# BERT-style masking for masked language modeling

Prepare a batch for BERT's masked language modeling objective (u16-l02).

```python
def mlm_mask(ids, mask_id, vocab_size, special_ids, generator, p=0.15):
    ...
    return inputs, labels
```

- `ids`: an `int64` tensor of shape `(B, T)`, token ids of padded sequences such as `[CLS] w1 w2 ... [SEP] [PAD] [PAD]`.
- `mask_id`: the id of `[MASK]`. `vocab_size`: the number of ids, $V$.
- `special_ids`: a 1-D `int64` tensor of ids that must **never** be chosen: `[PAD]`, `[CLS]`, `[SEP]`, `[MASK]` and the like.
- `generator`: a `torch.Generator`. Draw all randomness from it (pass `generator=generator` to every `torch.rand`, `torch.randint` or `torch.bernoulli` call), so the same seed always gives the same result.
- `p`: the selection probability.

The rules:

1. Choose each position whose id is not in `special_ids` independently with probability `p`.
2. At each chosen position, the input becomes `mask_id` with probability 0.8, a random id drawn uniformly from $0, \dots, V - 1$ with probability 0.1, and stays unchanged with probability 0.1. Positions that are not chosen keep their id.
3. `labels` holds the **original** id at every chosen position and $-100$ everywhere else, so that `F.cross_entropy` (whose `ignore_index` is $-100$) takes the loss at the chosen positions only.

Return `inputs` and `labels`, two new `int64` tensors of shape `(B, T)`. Do not modify `ids`.

**Example.** With ids 0 to 4 special (`[PAD]` = 0, `[UNK]` = 1, `[CLS]` = 2, `[SEP]` = 3, `[MASK]` = 4):

```python
ids = torch.tensor([[2, 17, 45, 9, 33, 81, 3, 0, 0]])
inputs, labels = mlm_mask(ids, 4, 100, torch.tensor([0, 1, 2, 3, 4]), torch.Generator().manual_seed(4), p=0.5)
# one correct output:
# inputs = [[2, 17,  4, 9,  4,  4,    3,    0,    0]]
# labels = [[-100, -100, 45, 9, 33, 81, -100, -100, -100]]
```

Four positions were chosen; three became `[MASK]` and the one holding 9 stayed unchanged but still has label 9. Your output may differ, since it depends on how you draw from the generator. The grader checks the rules exactly and the probabilities statistically: on a batch of 256 sequences of length 128 (24393 ordinary tokens), one correct solution chooses 15.0% of the ordinary tokens, and of those replaces 80.25% by `[MASK]`, 9.56% by a random id and leaves 10.19% unchanged. Every share must be within five standard deviations of its probability, and the random ids must spread evenly over the vocabulary.

**Restrictions.** Only `torch`. No loops or comprehensions: use boolean masks (`torch.isin`, `masked_fill`, `torch.where`, boolean indexing).

**Constraints.** $B \cdot T \le 2^{17}$, $V \le 10^5$.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem6_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem6_submission.py
import torch


def mlm_mask(ids, mask_id, vocab_size, special_ids, generator, p=0.15):
    # Your code
    return inputs, labels


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem6_submission as submission
importlib.reload(submission)
# e.g. submission.mlm_mask(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9XI132zaS/1ew6ruGzMqM7DS5rFL31kncbm7z9WI3u/ckPRYSIYlrimQJKrbX6/7t95sBQIKS7Njp3qXvyRIIDGYG841hr3pp0huK3nr/6V45eNrri95K1RJDV9tP6rTOFI29OP54uqfry0yJldRnab4Q86Li7yoRmcwXa7nAsyJRGR7S2kxpXeQOYDY44K3SfF0rjdH9J/iZpPN5Oltn9SVGHmNA5XVF33urbBUTdF5UJOtMxblcMS5lVUwztXoa6/V0lWqdYhfMkllWnKskTldlUdW0xahXF9Vs2Zvg6VTmuSLyRvZXnBVFSbPqaq2IVKXruE5XqljXMY0fDGhULiykSuYaJK9UFQE3Zg4BjzCeFN5vw5J4nmZZO5jqNGc0dLGuZkxFUaWLNJc8qZRVfWBRucZvLGYeXfWWssrBx6hkpoz5v7+orFSVFvVS1sIyQywqmdCgoV4Ev8yrYiXscjf68JcwGufj/MhOF6kWsyKvwfVH6zyt9aPv6c8PjyxU/C7T5IdHZjaQiMTrWiRqngKqII6J+Tqf1TgBLeh4Eh6MHxrcanmm8E2N8/akhDlM8QuGfonE8d+P3n54c3wislTXRJIyYAmYFnpZnOcCiz6ucxF8BheBVfhcnBC4WlRrbKs+q+qSF4G0H2WarSusrGSqlaCf4jytl0IKwNMkoudVWoNgFt8XKh8KLS/FOaF7LjWeFvmiL3KCKpL1qhTLNEkwPc3Lda2Je+YUcsvSWVFeNj9IfIi/9meO9ZcCUPOSRr8RL4tMTsXLDz9rISslNAlsRbzKCZUHWrw9EB+qoi90IbJiJjOCCHXSYrUGU6aKmAoWFaQ1KR5nKfFhCvqXOMq5nEHcItrpIBrQ4UqxWNP5B2ZnSOEiU3v1slIyEbpUODDMqor1YpldQlqyuQAuwGJWVMToShHTa5HOCUH8ZNWydFhlIaacfDg+fhX/ePTy9P1HcUi7E8HjfJZJ7E7HEBxfzFRJkhIOx7nAvxKPzCwIFDiukgCymPTFSi/cHOybF8TkPLEj9K893IDmtkDivAwu3FoYEm+NOyFSR290Lkg1dS3zmQou+uZ5dKpyXVSht5y3VfW6ysVFlMBezpZBGM3KNT75mIPQTFZMpnjN2x1XVVF5UAzJHqy8jKSWVSUvgXdLh17KUgWLou5DKHP6hHwejnuQbNjKcc9hpsHrel1mKiDKMT+MeKnFhXmKOW4SwQr7Yj7uXRHAaxy4NnuJKw27oy5KNashE1fe/Gts12I2ywp9G2Z9UdVFdriv9p72hbRfnzl8oVnnQLlBVsKjgNB5VkhCjMZ5z+4Dj5iFoY9IOjdfbyDHEGCnh7toc4+YvGYDHEiqYd7SGquBR5YFoYClICn0n53bZ/72ZEllypYwpzVpPm+AW0Gm884yy0Qww7KLPiy76MMXPFVV4BgzgpBbyQv6I6c6WIg9cR6GIcEGoek/lVAZ9GJAyrdDVbp8YkMnPssM9kEEmawWZHYVCay4wp9h9HhxHXbOXv26ltkdpLKrVGYqIZ0nLOn9kDkKJGRdV/b5uMfH0QLZIS6egLRz/s/FwhcNxj+2fAByW8rkc7WBQKcy3MQYwkDYMvEtDBjjKzz6Q9VBjWZhqHMYqY5ZRfg46Ju+8Tx4Q+9E2iVb+NMoWGEBh1HMQVccd9BZwPzRydldH7hJDxzULqaxzmUJT17//5rmrMjVPY0ytnRSeUEiyb69F8IlJqy9XXyCJJ2BkRS4hD4+XeI62BHAIFGqPDxFpOcJmEXwFOzbRG8bQNhxITQUEUx+dtFhPI4lkH0x/Xq+yzvx3UwxesH7fS3bJbGd4eiuKbAbSbOJDqbhLluD5a2d2bm8sRNT+5XPtqvZREHfWDtIdu6flYPDELzgpYhXaygFoptgnvfFQ5hTaMbDh2fn9M1hYkLHl/AAYp7zznMKUIkGxDiIAhdQL5kjZKwW6xWCcrgQUWZypiIXdbLXJ32CXRy1miVDjmclLaAdJ2YmojNyHnmwiZB5TEtSCDII1iEtVQhlVCXh3/6ZloFZw3t0BJztSSNc1ohcIrGBLCacA6bzlCJLCplbWr65SsUfxf71c3FeVGcU1UsWXkHHh3i0sZeWyUC+5TAF8WkeaEWhILC6K5ef+yyGdfVD7h2RNOeQ8IcasTcoIAZxrPugWYng3eRQtNV7gKgoBUOCtNJEEsfhb+VMzLABLQGRJuqmbIUDewoOkF3sTdcJLCkNcwyOsDylBGpdQp1EfZ7OFAEEh1J46GJuNRg4O45qFiEamSNeIvdNELBijWeROKUHaYVhEJM9oC3IL5jNiMMq6YhVPaBoEhAiZJfzmMGoylmbL4hSUu9ejBiltuGIpfdQ2EMUj4SfNrQiGZMoVqQNwYEvdzhEbPP9oYW0YYqmyGnO2iF50BdnB9itax8Z5f7GYIeSL7OC8SRGHDAbDrxhZgMytiAhF3sDO/x40yeI9aguCpYbJP9JPYwO5teI8iFl6rnNuhohZmGC2jEgsc4pn7+yrLHrOIdEYueOeMc/3hKZJdLgOURrQQGsSTKv7DkxINKOWxW0yhfQTpW4A2sTHFMgAbfnEuIXNxPHXLVZaynTIuYKQ6fK8RONQAOAWVs8SNloLNMsIa2bUWUECXNKukAz//vk/TvgmrPm6DoBhqyr5WW9xMjea+H2Ed/bEkecpNUP4vt2DzzEb2jlv3is/mEr5Yeyeek/DvaiztJpM2T+YCRa12nWDhfN13/ooi0SlLJe+st1ushlu0xf6o3ygvtRwS9M5eysGYEU51QsMLUGPUtBZq4ac4eBs7TeyxSmif3oabRvYzo9pDrRiup1fTYpyC6s83HGr5QpuLwY526PaJ5mkGj3Mxj3gHZRKcoAbaUFkWj08Of8LKcaji4yqqgUXAHAhmlJpxY9tJHiX44/HkNzLC+iD/gbxFRDQ1gZRjBctBxRXSkrYDbOwZWIJkeQVlXVwQDKgPCBwIQAuKv2xbUg8Q189a9yKI6/GxxwJezN347+5yR+cfTu3fErYHA17sW2ghjHRAwIc+dpfqoLNbPfEOebb7NiVQLXce96nL95//5D/O79q+MTgAsgK9GPBXSVvvwNgqvM1yN9mc+aB28QQ74EDPPrRHk/XiHIbH/9pHLyz0V1fFGGfonl1NRivlRliSUyvVVAMrZewd7AbalGYzlVdID8agRFNrN4tlSzMx3oakZHXMuuv/1o9J2iYTgr4mNxZpiz0gsw5jnMIkWApsbHaqPrB83pcDBkS4nkqOoqNWXFjosyTmg02RXN1pVCYMdcgpQgvwai3Tj05BL5+QXHoVSUU9vx4ciiPYS0IGlrsB+SmdS83KbIRW7szJWK6G9eXA/pO2Yj+bm2CNqKNLkE8CuCYYaebJSpiUejiUV0Kptx9pO1czlcDo3XmoF541PoQ2xqpXacRBwhI3Q3TaAqFIlVJlAwgAHBzA8oo6GgO51SRR7knNn6fUSCbTfgIrnH85WsV+vMH5lzdQSmmmwzmH8us7OADmPDdXthOk038mxSg82sogllaWLE6A63HZgtqSBeDiRPoqB8RaUF8zPCbmAkRY7BCpEnzBElc3xtwciagwiHu32jdxaRTBK7R7g9uT0bb16kyyyl46Y9R3v7kx0L28OL1mVCUfeOtX6SmN3Oxh9h9TapwYHirJiPtvAO8luF2sFLXmEZie+/g4ubHAS0DS54rOtwYJcEbCy9iXlftZTpvC/Tj5zubNK+IRCMBynanaC+8yzybQDT5Gaqdk/atWPrrDb3ZMWPZFmqPAlcxccsGveMyTPm/b+6/Nq1C3vBF2n+3nqxo/XiSJMPCk1tZWNBVNh5b2X9FiHjlkwbG3RP5ICYJ5AeSAoSLagbzb+LAoyWGJGnUVYImhf9o6D0lA1s4O0Thrutyx/Jo4AxDtTVg754YIA4jSIFfID9kPcuHlAp9np3PXEX/qd8p7mFvna4t8C+aW4QpzLXxNvZ0noIDgVbB8HlZyWRxP6d/4bPTTaanctLvWfuV5HJIC5MG0AklBZakWeXfbcnMpfW1QRGCkwRyYZSFHRcikwtkJqfU9gkeGYURaHnzBCUiaSgQzGAlynnl+YUAs9pfsvO0fPCBl3rfCGG/xKB50u/FZ240BMh7HBv0QEGWlj+EPxAhztkB5DD33PCeeH2aK5k21MG5l3iY6NEcVGaeLJbbaPpRsk2VO8uBFNGCbn4s3CwKWX19EYEJnKyO4wGE1+4d5B/Hxa0u95GPBu3LZJ59OsINmESC3uHUAyTGpNQV2qpck5im9iRV/176beUXd+Sqc+rFGCyy5gj2QAgkOjGlE11I/r3udpjPBOlZ1XKWYUoIao1dX/Ym2iekLrra0rYoK6K5mm+rO9WmKZUVnF5a4S8Gd/ruJ4GKorj5gFSPhtsEnBEmnN26nPaByBwUnNKP5WL+Sjza0iwUek5lUyoYEVWlmuTVujwSeGYF7V3hugoWGBoNl+ntehbZtKNCd+QKO9+hKL/6yve9ZrmO2avEIkHTS4GS7uZ5VImK6vF51ETIjpSbpx5MPFSYsspCsAPRTPlsQvXIfcYp4JDlBUy0UHASDwisSGd4FJEj+DBZ1ElI3DGDukTQbTI+BMcN7iaeCi29cIkjGPbbDPm1hrd/jSFGSOyNpMmSaSBdwWOwIc/aqFNCJ0bUtFG0SmCJRXUI4PVhL0F/WItmfjhrEvpXDyrue64ta1vJbhqETA7qTlEB2a67+WNlLiS5xxIp0V0UlP55PV7xzxT5InMn8D+Onn909Gbj2/7Nku3U2HOYi7uultgz5Z1e5aIlwcDh0s3O+YmmLZYheNM0kpB+cxhBMB0M8xyOKo6pS0qh+br09dvjz/GH4+P3vQb7DbCHF0qkp5uKSyi0ZhqM6ayQ9VJvjLxKPJavYxotIU5HJNnqjb2W0+3t7OweEPaOqCPzYXQGDNPj+gxJwwTI/jbJLEOqSqiGlBslpHlCb+CcYNubcKWXTolCSuIVjkmxpoZGskCs1Vji1sXxRnfKYgrdx7DaEAVYGe67C4vpFZNhcjUQNqS2Is3x4PBfovAvci4CWN7McRo0k0MdYgV9qBsEHSLP2rUegv0LkY5u0L7Qp5JovgmPkAC/t1gMBhOvlqRuS2xW8fhIb+/w8xxd8XuRhO09M2j8MuH2/IqKZQJmkyzHbwLgbj28/Z7krC4n06Oe6bpj7SwcRlNH2BDt+0j/LLuLTzlW9ykSwaanWXb/Wxhy2V5ZgpZBtc66IJ4a/bI11DQECBRmRvDTt/Itn+WlXZ7RGmtVjrg7pm86xXYqrp7f7pAkUAjmId+wQtWgi4hOSShHTeCZ3bHJP3VGnbLCAVFK9xFkDvSNgwumec0Xyv//GbGwRp7OGQgoyfDCew33wsDW1ulFmwexz0L2k32dqOnVA2mtGdXmmLtrbn0oaFBNLi+z53YduPDV3id3+t57OXctlX+KpNmD2HUco40lQLv7d6NLQveLjaxh9FxNtZZkS/cvcymzRZBmpsuM47m+5RAzGUl3M1g6BsBuzlfbmzUtH8nwQ3OFF7v6FW5ly/5d+Hi9xnwfUVyRx/SwmoEnOBVkOIkuOmiti8e73JvNpyduISsagzbvV1Rs8DEqoeUOgZt7HpbVMpGZVoUWbCFmHlGoOqO5BK4ugOuWeESlBtdCrc1z4VLeIx1i2PKb+iibOiuByjdMTe7jbOgO11bPkPOv2xvME3j0U2XdX2+2O77zTWERNMxDhtvbbW6kKuSbB64+uHoVV/8/O6vffHyzUlfnBx/6Iu3Ryd/xXSc5z4i5D69aPAd9yy/fH30hqwaV55q7nAKRjdBmHhXYlOqbvE1dl+86IvTvvgEiyQT0yvUSaNHgDLBFmcAL0aANqHEOafZCcI46rGmPm96LtLEFBGeRNGnvf1O7rxo8GzuAIMwWsmcupQ0N/qaS3V2QYluptP1O53qE8YxIGzhKBYOyOHC3fSofFEvt9edikePiGmn1LzDADaWk8MDMSZbNmvna8iemXnqRKtoQUvT3HEajijf6wunDUB7NIQNIEUA17Zp4fw6YFCHIrAY0xKCM4HO7od9D4WY7rSClPqFwHc8wtfwNqg/HIpNoDfAg5B04NnaAAa8loh1bsI/XgHmG3mBRUdEoZouX/DyPmdr7kD9jrNMrqaJFBdDMnqRe3GGWhRJbHlnK+t8cMDgsOwgv9kgii36pmXcWBLwhMZC4vqBaRllCq6HbUMcTyfXRa9JWAp12MbHnXEQwCWpbjh1QaYpoFYCnkzxiPkGZDFqltKoBb7Vk3ZLv6ipMHOL74Vt8W374sEJO9bfiBl8Uq8Iy2v3JgaRzKDptrvTXdxC29FEfBEl3GN76E4cKvb0u/5tG9kZ1DF8ZZdfb7bidNnbeZGgUnFxFpz1KQouTY/whgDyJBzJGWJ87i+HnXTRdJHhwRPxkC13pH+t6qDEr2AfylaG/oKQ7IPa+5MnUtSlboDz5O+J5myDVtv3jDBofzAAZJ4/jPbn1//BfQT5NZmOlOvZXhOynJIe2DWlmd99UYFKRHFFib3R2C6LjMHeYMRsWWhWRiulfzgUe9jBcgk7U6Ngc3AQNmtbjHJ1XiOgl1YYmv+agCPagqeG67zgLThDTJUe2teZHMnmPapzaXJBix8LxlJ+VgaQAXBLi5cVZF3Yt7KobFvA1+qztITHrbtvQPi9vAbRkdl3wibD/eioyg7iDJJFZhoU3398/dPrd3C2zs9R/diS0xyvCEhJDTuIC6aGS24l3HhJA7wI7Opv3cGEEdX+Njgtm2ODolqP2zKX+GrAPOfrKUzCBEmvkxl3vZKX0MCb+WNkavRbh0G/NRzqoGLmtm2+NZC79Zi/cKLi9tMOPOhmT3MFZ1phtRUdezNDPA+99xXShc28KQAIfms4rNerpj6c28dWzP1HjdWBYXDAWNvGPUubkSx/b59ZfWuSoP8bJ9LgeEFpUbpgHPgMdsnoBjZMzAWZXvKLjho2jIPoGczXINqHRftEeNr8OqF360Yc/W2iKFf0qsemDDcIFtwajCjlgswIbwhZNb8I83AHcrymi9a+M7YHhFm4iZoUprvSsOeeKO5gDGO2iUGHMZma12KdWzG+547uPYYRUzpprbUfxnPuPvTDNBNtI3w/ePIUrubgGT4Gg8FNoUU37qKZHPiDiCemgc9mC6Gz+l/wFAbCruUd9M1JxEZQ4zWypYJa7VpiPpErde7Epwwx1ZP9A0vep7uR9YnTGIeU3Y0RabmdU+/PXRzhrWBMP3nj96ZpziMBgX8I5lB+ABirNDex8+H+wHu/4Kxt5sa4F7F1pM/sMTojo0nRJmCHjBaLly/jWsxVlonUJEtXZ0DhE2GwP7iOoqvgjKKQsB2knADRUn+HKW3gWoUyLfV6WazhtBB3OWZwYX0gokh8LmZyGvObdgC7LQEs1XF5kwB/5wvws7vK7zMSvifEjMdcwaNENnp8d+l9ZoR3a3EHc6QT1jvehLzDm0LRxwSSsl0uIt6NDF7zn62oIfCxO96dlMe+Hm4A6JCDmCu2Tmubnm62P2qTfE788QEnYkZN+uSeTSZ3NjjgOiH6J8JTGgZymmBjESvI5I6/FHvdkH6y0/TbZvx4xKR2vsh/CYuhsH1GNuKxFty8h26oJH9Mr9RweMYdOCae4Y057902iKqsimQ9I/d/k1iBRY8PjFR5Jt39nxTaBNglJPIm++5JRrtvmqV1y+JtqO7VjenvgLp1cHI0gARN8bl9QHK0z8/2JxvnU58XbVFFm8I5Bc70lhXXDcSCAn76/2UoaoG3JTwkDFTe8+oxdL1mcghr3egCg5HtXV//L/Wjwkk=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
